# ZemaVoice Stage 11 — OmniVoice Amharic Test

**Goal:** Test whether OmniVoice Amharic produces a more natural youthful *character voice* than Stage 10. First generate baseline speech; optionally upload a **consented, authorized** reference WAV of an adult performer using a youthful character voice. Do not use a child's recording without a parent/guardian's permission, and do not clone anyone without permission.

**Important:** The public model card supplies sample code, but the exact package/model combination has **not yet been validated in this notebook**. If installation or model loading fails, share the error; the diagnostic cells will show it. The model card lists Apache-2.0; verify model and any dependency licenses before other use.

Set **Runtime → Change runtime type → T4 GPU** before running. A free Colab GPU may not always be available.


In [ ]:
import sys, subprocess
print('Python:',sys.version)
import torch
print('Torch:',torch.__version__,'CUDA available:',torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: For this test select Runtime > Change runtime type > T4 GPU, then rerun.')


In [ ]:
import subprocess,sys
result=subprocess.run([sys.executable,'-m','pip','install','-q','omnivoice','soundfile','huggingface_hub','ipywidgets'],capture_output=True,text=True)
print('Install return code:',result.returncode)
if result.returncode:
    print(result.stderr[-5000:])
    raise RuntimeError('OmniVoice package installation failed. Copy the error above and send it to ChatGPT; do not proceed.')
print('Packages installed.')


In [ ]:
import torch, soundfile as sf, numpy as np
from IPython.display import Audio,display
from pathlib import Path
try:
    from omnivoice import OmniVoice, OmniVoiceGenerationConfig
    print('OmniVoice Python import successful.')
except Exception as e:
    print('OmniVoice import failed:',repr(e))
    raise
assert torch.cuda.is_available(),'Please enable T4 GPU in Runtime > Change runtime type, then rerun.'


In [ ]:
# The model card publishes one repository ID for its example and another for its own page.
# Try the ID shown in the example first; fall back to the visible model repository.
model=None
errors=[]
for model_id in ['african-low-resource/omnivoice-amharic','gheero-Leyu/amharic-omnivoice-tts']:
    try:
        print('Trying model:',model_id)
        model=OmniVoice.from_pretrained(model_id,device_map='cuda:0',dtype=torch.float16)
        print('Loaded:',model_id)
        break
    except Exception as e:
        print('Load failed:',type(e).__name__,str(e)[:600])
        errors.append((model_id,repr(e)))
if model is None:
    raise RuntimeError('Neither published model ID loaded. Send the error messages above to ChatGPT. Details: '+repr(errors)[:1200])


In [ ]:
amharic_text='ሰላም ጓደኞቼ! እንዴት ናችሁ? ዛሬ አብረን እንጫወታለን!'
print('Testing:',amharic_text)
result=model.generate(text=amharic_text,language='Amharic',generation_config=OmniVoiceGenerationConfig(num_step=32,guidance_scale=2.0))
wave=result[0]
if isinstance(wave,torch.Tensor): wave=wave.detach().cpu().float().numpy()
wave=np.asarray(wave,dtype=np.float32).squeeze()
assert wave.ndim==1 and len(wave)>0,'Generated audio has unexpected shape.'
sf.write('/content/zemavoice_stage11_baseline.wav',wave,24000)
print('Saved baseline:',round(len(wave)/24000,2),'seconds')
display(Audio(wave,rate=24000))


## Optional: youthful character voice reference
Use a **10–20 second** clean WAV recording of a person who has agreed to have their voice used for synthesis. For a youthful style, a consenting adult voice actor performing a playful character voice is a practical first test. The model may not preserve age/style reliably; evaluate the actual output. Skip this section if you don't have an authorized recording.


In [ ]:
from google.colab import files
uploaded=files.upload()
assert uploaded,'Please select a WAV file, or skip the optional voice-cloning section.'
ref_path=next(iter(uploaded))
print('Uploaded reference:',ref_path)
ref_audio,ref_sr=sf.read(ref_path)
print('Reference duration:',round(len(ref_audio)/ref_sr,2),'seconds | sample rate:',ref_sr)
if len(ref_audio)/ref_sr<5:
    print('WARNING: A longer clean reference (about 10 seconds) is recommended.')


In [ ]:
prompt=model.create_voice_clone_prompt(ref_audio=ref_path,ref_text=None)
cloned=model.generate(text=amharic_text,language='Amharic',voice_clone_prompt=prompt,generation_config=OmniVoiceGenerationConfig(num_step=32,guidance_scale=2.0))
y=cloned[0]
if isinstance(y,torch.Tensor): y=y.detach().cpu().float().numpy()
y=np.asarray(y,dtype=np.float32).squeeze()
assert y.ndim==1 and len(y)>0,'Cloned audio has unexpected shape.'
sf.write('/content/zemavoice_stage11_character_test.wav',y,24000)
print('Saved character test:',round(len(y)/24000,2),'seconds')
display(Audio(y,rate=24000))


## Download results
Run this cell after baseline generation; it downloads any generated WAV files. If optional voice cloning failed, the baseline can still be downloaded.


In [ ]:
from google.colab import files
from pathlib import Path
for name in ['zemavoice_stage11_baseline.wav','zemavoice_stage11_character_test.wav']:
    path=Path('/content')/name
    if path.exists():
        print('Downloading:',name)
        files.download(str(path))
